In [19]:
# Import required libraries
import pandas as pd
import numpy as np
import torch # X-ANFIS uses PyTorch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import warnings
import glob
import os
import collections.abc
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
# import uk_holidays # Removed this library

# Import X-ANFIS components
# Make sure you have installed xanfis: pip install xanfis
try:
    # *** Import GdAnfisRegressor instead of AnfisRegressor ***
    from xanfis.models.gd_anfis import GdAnfisRegressor
    # from xanfis.models.classic_anfis import AnfisRegressor
    # from xanfis.models.bio_anfis import BioAnfisRegressor
except ImportError:
    print("Error: xanfis library not found. Please install it using 'pip install xanfis'")
    exit()

# --- Configuration: File Paths ---
DATA_DIR = "."
# *** Reinstated PATH_BANK_HOLIDAYS ***
PATH_BANK_HOLIDAYS = os.path.join(DATA_DIR, "uk_bank_holidays.csv")
PATH_INFO_HOUSEHOLDS = os.path.join(DATA_DIR, "informations_households.csv")
PATH_WEATHER = os.path.join(DATA_DIR, "weather_daily_darksky.csv")
OUTPUT_CSV_PATH = os.path.join(DATA_DIR, "daily_merged_predictions_xanfis.csv") # Updated output filename
# MF_PLOT_DIR = "mf_plots" # X-ANFIS handles MFs internally, plotting might need custom code if desired
DIST_PLOT_DIR = "distribution_plots" # Keep distribution plots
DAILY_DATA_FOLDER = "daily_dataset"
DAILY_DATA_PATTERN = os.path.join(DATA_DIR, DAILY_DATA_FOLDER, "block_*.csv")

# --- Configuration: Features and Columns ---
# Define features to be used as input for ANFIS
# Make sure these columns exist after preprocessing and normalization!
ANFIS_INPUT_FEATURES = [
    "temperatureMax", "temperatureMin", "dewPoint", "humidity",
    "cloudCover", "windSpeed", "pressure",
    "month", # Keep month as a feature
    "is_holiday", # Keep holiday flag as a feature
    "previous_day_energy_normalized" # Use the lagged energy
]
TARGET_FEATURE = 'actual_energy_normalized' # Target variable for prediction

# Original weather features list (used for loading/cleaning)
WEATHER_FEATURES_RAW = [
    "temperatureMax", "temperatureMin", "dewPoint", "humidity",
    "cloudCover", "windSpeed", "pressure"]
DAILY_DATASET_NAN_DROP_COLS = ['energy_median', 'energy_mean', 'energy_max', 'energy_sum']
WEATHER_NAN_DROP_COLS = WEATHER_FEATURES_RAW.copy() # Use a copy for modification

# --- Configuration: Plotting ---
SCATTER_PLOT_SAMPLE_SIZE = 5000
LINE_PLOT_DAYS = 90

# --- Configuration: X-ANFIS Model ---
# *** Using num_rules directly based on documentation ***
ANFIS_NUM_RULES = 25 # Keep number of rules from previous attempt
ANFIS_EPOCHS = 300 # Keep increased epochs
# *** Define optimizer parameters (learning rate) ***
ANFIS_OPTIM_PARAMS = {'lr': 0.001} # Keep adjusted learning rate

# =============================================================================
# Data Loading and Preprocessing Functions
# =============================================================================
def load_data():
    """Loads all necessary datasets."""
    print("Loading datasets...")
    try:
        # *** Load bank holidays from CSV ***
        df_bank_holidays = pd.read_csv(PATH_BANK_HOLIDAYS)
        df_info_households = pd.read_csv(PATH_INFO_HOUSEHOLDS) # Load but don't merge later
        df_weather = pd.read_csv(PATH_WEATHER)
        print(f"Looking for daily energy files matching: {DAILY_DATA_PATTERN}")
        block_files = glob.glob(DAILY_DATA_PATTERN)
        if not block_files: print(f"Error: No files found matching the pattern '{DAILY_DATA_PATTERN}'."); exit()
        print(f"Found {len(block_files)} daily energy block files. Loading and concatenating...")
        all_daily_data = [pd.read_csv(f) for f in tqdm(block_files, desc="Loading block files")]
        all_daily_data = [df for df in all_daily_data if not df.empty and all(col in df.columns for col in ['LCLid', 'day'] + DAILY_DATASET_NAN_DROP_COLS)]
        if not all_daily_data: print("Error: No valid daily block files could be loaded or processed. Exiting."); exit()
        df_daily_dataset = pd.concat(all_daily_data, ignore_index=True)
        print(f"Combined daily dataset loaded successfully ({len(df_daily_dataset)} rows).")
        print("Core datasets loaded successfully.")
        # *** Return df_bank_holidays loaded from CSV ***
        return df_daily_dataset, df_bank_holidays, df_info_households, df_weather
    except FileNotFoundError as e: print(f"Error loading file: {e}. Check paths: {PATH_BANK_HOLIDAYS}, {PATH_INFO_HOUSEHOLDS}, {PATH_WEATHER}"); exit()
    except Exception as e: print(f"An unexpected error occurred during file loading: {e}"); exit()

def handle_missing_values(df_daily_dataset, df_weather):
    """Handles missing values in daily and weather dataframes."""
    # Need to modify global ANFIS_INPUT_FEATURES if columns are missing
    global WEATHER_NAN_DROP_COLS, WEATHER_FEATURES_RAW, ANFIS_INPUT_FEATURES
    print("\nHandling missing values...")
    initial_daily_rows = len(df_daily_dataset)
    df_daily_dataset.dropna(subset=DAILY_DATASET_NAN_DROP_COLS, inplace=True)
    print(f"Dropped {initial_daily_rows - len(df_daily_dataset)} rows with missing energy data from original daily data.")

    initial_weather_rows = len(df_weather)
    # Check for missing weather columns and update lists if necessary
    original_weather_cols = set(WEATHER_FEATURES_RAW)
    missing_cols = original_weather_cols - set(df_weather.columns)
    if missing_cols:
        print(f"Warning: Columns not found in weather data: {missing_cols}. Removing from NaN check and feature lists.")
        WEATHER_NAN_DROP_COLS = [col for col in WEATHER_NAN_DROP_COLS if col not in missing_cols]
        WEATHER_FEATURES_RAW = [col for col in WEATHER_FEATURES_RAW if col not in missing_cols]
        # Update the global ANFIS_INPUT_FEATURES list
        ANFIS_INPUT_FEATURES = [col for col in ANFIS_INPUT_FEATURES if col not in missing_cols]
        print(f"Updated global ANFIS input features: {ANFIS_INPUT_FEATURES}")

    valid_weather_nan_drop_cols = [col for col in WEATHER_NAN_DROP_COLS if col in df_weather.columns]
    if len(valid_weather_nan_drop_cols) < len(WEATHER_NAN_DROP_COLS):
         print(f"Warning: Some weather columns specified for NaN drop were not found or already removed: {set(WEATHER_NAN_DROP_COLS) - set(valid_weather_nan_drop_cols)}")

    if valid_weather_nan_drop_cols:
        df_weather.dropna(subset=valid_weather_nan_drop_cols, inplace=True)
        print(f"Dropped {initial_weather_rows - len(df_weather)} rows with missing weather data based on columns: {valid_weather_nan_drop_cols}.")
    else:
        print("Skipping weather NaN drop as no valid key columns were found.")
    return df_daily_dataset, df_weather

# *** Updated function signature and logic ***
def process_dates_holidays(df_daily_dataset, df_weather, df_bank_holidays):
    """Converts date columns and prepares holiday flags using the loaded CSV."""
    print("\nProcessing dates and holidays...")
    # Process weather dates
    if 'time' in df_weather.columns and pd.api.types.is_numeric_dtype(df_weather['time']):
        df_weather['time'] = pd.to_datetime(df_weather['time'], unit='s', errors='coerce')
    elif 'time' in df_weather.columns:
         df_weather['time'] = pd.to_datetime(df_weather['time'], errors='coerce')
    else: print("Warning: 'time' column not found in weather data.")
    df_weather.dropna(subset=['time'], inplace=True)
    df_weather['day'] = df_weather['time'].dt.normalize()

    # Process daily dataset dates
    df_daily_dataset['day'] = pd.to_datetime(df_daily_dataset['day'], errors='coerce')
    df_daily_dataset.dropna(subset=['day'], inplace=True)

    # Process bank holidays DataFrame loaded from CSV
    print("Processing bank holidays from CSV...")
    # Find the correct date column name in the holiday CSV
    holiday_col_name = None
    potential_date_cols = [col for col in df_bank_holidays.columns if 'date' in col.lower() or 'day' in col.lower()]
    if 'Bank holidays' in df_bank_holidays.columns: # Specific name from original code
        holiday_col_name = 'Bank holidays'
    elif potential_date_cols:
        holiday_col_name = potential_date_cols[0] # Take the first potential match
        print(f"Assuming '{holiday_col_name}' is the date column in the holiday CSV.")
    else:
        print(f"Error: Date column not found in the bank holiday CSV ({PATH_BANK_HOLIDAYS}). Please check the CSV header.")
        # Create an empty DataFrame to avoid errors later, but holidays will be missing
        df_bank_holidays = pd.DataFrame(columns=['day', 'is_holiday'])
        return df_daily_dataset, df_weather, df_bank_holidays # Return early

    # Rename and process the holiday dates
    if holiday_col_name:
        df_bank_holidays.rename(columns={holiday_col_name: 'day'}, inplace=True)
        try:
            # Try specific format first, then generic
            df_bank_holidays['day'] = pd.to_datetime(df_bank_holidays['day'], format='%Y-%m-%d', errors='coerce')
        except ValueError:
            df_bank_holidays['day'] = pd.to_datetime(df_bank_holidays['day'], errors='coerce')

        df_bank_holidays.dropna(subset=['day'], inplace=True)
        df_bank_holidays['is_holiday'] = 1
        df_bank_holidays = df_bank_holidays[['day', 'is_holiday']].drop_duplicates(subset='day')
        print(f"Processed {len(df_bank_holidays)} unique bank holiday dates from CSV.")
    else:
        # If no holiday column was identified, ensure df_bank_holidays is empty with correct columns
         df_bank_holidays = pd.DataFrame(columns=['day', 'is_holiday'])


    print("Date conversions complete.")
    return df_daily_dataset, df_weather, df_bank_holidays

def aggregate_and_merge(df_daily_dataset, df_weather, df_bank_holidays):
    """Calculates daily average energy and merges with weather/holidays."""
    print("\nAggregating daily energy and merging datasets...")
    print("Calculating daily average energy per household...")
    df_daily_agg = df_daily_dataset.groupby('day').agg(
        total_energy_sum=('energy_sum', 'sum'),
        household_count=('LCLid', 'nunique')
    ).reset_index()
    # Avoid division by zero if household_count is 0 for any day
    df_daily_agg['avg_energy_per_household'] = np.where(
        df_daily_agg['household_count'] > 0,
        df_daily_agg['total_energy_sum'] / df_daily_agg['household_count'],
        0 # Or np.nan if you prefer to drop these rows later
    )
    # Replace potential infinities (though unlikely with the check above) and NaNs
    df_daily_agg['avg_energy_per_household'].replace([np.inf, -np.inf], np.nan, inplace=True)
    df_daily_agg.dropna(subset=['avg_energy_per_household'], inplace=True)
    print(f"Daily average energy calculated for {len(df_daily_agg)} days.")

    # Ensure weather features to merge actually exist in df_weather
    weather_cols_to_merge = ['day'] + [col for col in WEATHER_FEATURES_RAW if col in df_weather.columns]
    df_weather_daily = df_weather[weather_cols_to_merge].drop_duplicates(subset='day', keep='first')

    print("Merging daily aggregated energy with weather and holidays...")
    df_merged = pd.merge(df_daily_agg, df_weather_daily, on='day', how='inner')
    print(f"Rows after merging aggregated energy with weather: {len(df_merged)}")

    # Merge holidays (df_bank_holidays now comes from CSV processing)
    df_merged = pd.merge(df_merged, df_bank_holidays, on='day', how='left')
    df_merged['is_holiday'] = df_merged['is_holiday'].fillna(0).astype(int)
    print(f"Rows after adding holidays: {len(df_merged)}")

    # Add month column
    df_merged['month'] = df_merged['day'].dt.month
    print("Added 'month' column.")

    if df_merged.empty: print("\nError: DataFrame empty after merging."); exit()
    print(f"\nTotal rows in final merged DataFrame (Daily Level): {len(df_merged)}")
    print("Columns:", df_merged.columns.tolist())
    return df_merged

def normalize_features(df_merged):
    """Normalizes features required for ANFIS input and the target variable."""
    print("\nNormalizing features...")
    df_normalized = df_merged.copy()

    # --- Normalize Target Feature ('avg_energy_per_household') FIRST ---
    # This is needed to create the lagged feature before scaling all inputs
    print(f"Normalizing target feature '{TARGET_FEATURE.replace('_normalized', '')}'...")
    actual_energy_col = 'avg_energy_per_household'
    if actual_energy_col in df_normalized.columns:
        # Ensure numeric and handle NaNs/Infs
        df_normalized[actual_energy_col] = pd.to_numeric(df_normalized[actual_energy_col], errors='coerce')
        if df_normalized[actual_energy_col].isnull().any():
            median_val = df_normalized[actual_energy_col].median()
            df_normalized[actual_energy_col].fillna(median_val, inplace=True)
            print(f"Filled NaNs in '{actual_energy_col}' with median ({median_val}).")
        df_normalized[actual_energy_col].replace([np.inf, -np.inf], np.nan, inplace=True)
        df_normalized[actual_energy_col].fillna(df_normalized[actual_energy_col].median(), inplace=True) # Fill again if Infs were replaced

        # Apply MinMaxScaler
        scaler_target = MinMaxScaler()
        df_normalized[TARGET_FEATURE] = scaler_target.fit_transform(df_normalized[[actual_energy_col]])
        print(f"Target feature '{actual_energy_col}' normalized to '{TARGET_FEATURE}'.")
    else:
        print(f"Warning: Target column '{actual_energy_col}' not found. Cannot normalize.")
        df_normalized[TARGET_FEATURE] = np.nan

    # --- Create Lagged Normalized Energy Feature ---
    lagged_col = 'previous_day_energy_normalized'
    if TARGET_FEATURE in df_normalized.columns:
        df_normalized[lagged_col] = df_normalized[TARGET_FEATURE].shift(1)
        # Fill initial NaN using backward fill (takes the next day's value)
        try:
            df_normalized[lagged_col] = df_normalized[lagged_col].bfill(limit=1)
        except AttributeError: # Older pandas might not have limit in bfill directly
             df_normalized[lagged_col].fillna(method='bfill', limit=1, inplace=True)

        # Fill any remaining NaNs (e.g., if the first value was NaN after bfill) with the mean
        if df_normalized[lagged_col].isnull().any():
             mean_normalized_energy = df_normalized[TARGET_FEATURE].mean()
             df_normalized[lagged_col].fillna(mean_normalized_energy, inplace=True)
             print(f"Filled remaining NaNs in '{lagged_col}' with mean ({mean_normalized_energy:.4f}).")
        print(f"Created lagged feature '{lagged_col}'.")
    else:
        print(f"Warning: Cannot create lagged feature as '{TARGET_FEATURE}' is missing.")
        df_normalized[lagged_col] = np.nan


    # --- Normalize ALL ANFIS Input Features (including month, holiday, lagged) ---
    # *** Scaling ALL input features now ***
    features_to_scale = [col for col in ANFIS_INPUT_FEATURES if col in df_normalized.columns]
    print(f"Input features to normalize (0-1 scale): {features_to_scale}")

    if features_to_scale:
        # Impute NaNs before scaling (using median) - should mostly be handled already, but as safeguard
        for col in features_to_scale:
            if df_normalized[col].isnull().any():
                median_val = df_normalized[col].median()
                # Avoid filling NaNs in boolean-like 'is_holiday' with median if it's not 0 or 1
                if col == 'is_holiday' and median_val not in [0, 1]:
                    fill_val = 0 # Default to non-holiday if median is weird
                else:
                    fill_val = median_val
                df_normalized[col].fillna(fill_val, inplace=True)
                print(f"Filled NaNs in '{col}' with {fill_val}.")

        # Apply MinMaxScaler to all selected input features
        scaler_inputs = MinMaxScaler()
        df_normalized[features_to_scale] = scaler_inputs.fit_transform(df_normalized[features_to_scale])
        print("All input features normalization complete.")
    else:
        print("No input features identified for normalization.")


    # --- Final Check for NaNs in ANFIS features and target ---
    final_features_to_check = ANFIS_INPUT_FEATURES + [TARGET_FEATURE]
    nan_counts = df_normalized[final_features_to_check].isnull().sum()
    if nan_counts.sum() > 0:
        print("\nWarning: NaNs detected in final features/target after processing:")
        print(nan_counts[nan_counts > 0])
        print("Attempting to drop rows with NaNs in these critical columns...")
        df_normalized.dropna(subset=final_features_to_check, inplace=True)
        print(f"Rows remaining after final NaN drop: {len(df_normalized)}")
    else:
        print("\nNo NaNs found in final ANFIS input features or target column.")


    return df_normalized # Return the DataFrame with normalized columns

# *** Visualize Data Distributions *** (Keep this)
def visualize_data_distributions(df, features_to_plot):
    """Generates and saves histograms for specified features."""
    print("\nGenerating Data Distribution plots...")
    if not os.path.exists(DIST_PLOT_DIR): os.makedirs(DIST_PLOT_DIR)

    for col in features_to_plot:
        if col in df.columns and pd.api.types.is_numeric_dtype(df[col]):
            try:
                plt.figure(figsize=(8, 5))
                # Use more bins for potentially discrete features like month/holiday after scaling
                bins = 50 if col in ['month', 'is_holiday'] else 30
                sns.histplot(df[col].dropna(), kde=False, bins=bins) # Turn off KDE for discrete/scaled discrete
                plt.title(f'Distribution of {col}') # Title reflects actual column name
                plt.xlabel('Value (Scaled)' if col in ANFIS_INPUT_FEATURES else 'Value')
                plt.ylabel('Frequency')
                plt.grid(True, axis='y')
                plot_filename = os.path.join(DIST_PLOT_DIR, f"dist_{col}.png")
                plt.savefig(plot_filename)
                plt.close()
                print(f"Saved distribution plot for {col} to {plot_filename}")
            except Exception as e:
                print(f"Could not plot distribution for {col}: {e}")
        else:
            print(f"Skipping distribution plot for {col} (not found or not numeric).")
    print("Finished generating distribution plots.")


# =============================================================================
# X-ANFIS Model Training and Prediction
# =============================================================================

def train_predict_xanfis(df_processed):
    """Trains an X-ANFIS model and adds predictions to the DataFrame."""
    # Global config variables
    global ANFIS_NUM_RULES, ANFIS_EPOCHS, ANFIS_OPTIM_PARAMS
    print("\n--- Training X-ANFIS Model ---")

    # 1. Prepare Data for Scikit-learn/PyTorch
    if TARGET_FEATURE not in df_processed.columns or df_processed[TARGET_FEATURE].isnull().all():
        print(f"Error: Target feature '{TARGET_FEATURE}' is missing or all NaN. Cannot train.")
        return df_processed.assign(predicted_energy=np.nan) # Return with NaN predictions

    # Ensure all input features exist
    missing_inputs = [col for col in ANFIS_INPUT_FEATURES if col not in df_processed.columns]
    if missing_inputs:
        print(f"Error: Missing required input features for ANFIS: {missing_inputs}")
        return df_processed.assign(predicted_energy=np.nan)

    X = df_processed[ANFIS_INPUT_FEATURES].values
    y = df_processed[TARGET_FEATURE].values

    # Check for NaNs/Infs again just before splitting
    if np.isnan(X).any() or np.isinf(X).any() or np.isnan(y).any() or np.isinf(y).any():
        print("Error: NaNs or Infs detected in X or y immediately before splitting. Check normalization/imputation.")
        # Optionally, try dropping again, but it's better to fix the source
        rows_before = len(df_processed)
        df_processed.dropna(subset=ANFIS_INPUT_FEATURES + [TARGET_FEATURE], inplace=True)
        X = df_processed[ANFIS_INPUT_FEATURES].values
        y = df_processed[TARGET_FEATURE].values
        print(f"Dropped {rows_before - len(df_processed)} rows due to NaNs/Infs.")
        if len(df_processed) == 0:
             print("Error: No data left after final NaN/Inf drop.")
             return df_processed.assign(predicted_energy=np.nan)


    # 2. Split Data into Training and Testing sets
    try:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
        print(f"Data split into training ({len(X_train)} samples) and testing ({len(X_test)} samples).")
    except ValueError as e:
        print(f"Error during train_test_split: {e}. Check data shapes and content.")
        print(f"X shape: {X.shape}, y shape: {y.shape}")
        print(f"NaNs in X: {np.isnan(X).sum()}, NaNs in y: {np.isnan(y).sum()}")
        return df_processed.assign(predicted_energy=np.nan)


    # 3. Define and Train the ANFIS Model
    print(f"Initializing GdAnfisRegressor with {X_train.shape[1]} inputs and num_rules={ANFIS_NUM_RULES}.") # Changed print statement

    try:
        # *** Use GdAnfisRegressor for gradient-based learning ***
        model = GdAnfisRegressor(
            num_rules=ANFIS_NUM_RULES,
            mf_class='Gaussian', # Default, but explicit here
            epochs=ANFIS_EPOCHS,
            batch_size=32, # Default is 16, can adjust
            optim='Adam', # Default optimizer
            optim_params=ANFIS_OPTIM_PARAMS, # Pass learning rate dict
            early_stopping=True,
            n_patience=20, # Increased patience
            epsilon=0.0001, # Lower epsilon for early stopping
            valid_rate=0.1,
            seed=42,
            verbose=True
        )

        # Train the model
        print(f"Training ANFIS model for up to {ANFIS_EPOCHS} epochs (with early stopping)...")
        # Reshape y_train to be (n_samples, 1) as often required by regressors
        model.fit(X_train, y_train.reshape(-1, 1))
        print("ANFIS model training complete.")

    except Exception as e:
        print(f"Error during ANFIS model initialization or training: {e}")
        import traceback
        traceback.print_exc() # Print detailed traceback
        return df_processed.assign(predicted_energy=np.nan)

    # 4. Make Predictions on the Full Dataset
    print("Making predictions on the full dataset...")
    try:
        # Predict on the original X data (before split) to get predictions for all rows
        full_predictions = model.predict(X)
        # Add predictions to the original DataFrame
        # Ensure the index aligns correctly if rows were dropped due to NaNs earlier
        # The prediction output might be 2D, flatten if necessary
        df_processed['predicted_energy'] = full_predictions.flatten()
        print("Predictions added to DataFrame.")

        # Optional: Evaluate on Test Set
        print("\n--- Evaluating on Test Set ---")
        test_predictions = model.predict(X_test)
        # Calculate metrics (e.g., RMSE, MAE)
        from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
        # Ensure y_test is also 1D for comparison if predictions are flattened
        rmse = np.sqrt(mean_squared_error(y_test, test_predictions.flatten()))
        mae = mean_absolute_error(y_test, test_predictions.flatten())
        r2 = r2_score(y_test, test_predictions.flatten())
        print(f"Test Set RMSE: {rmse:.4f}")
        print(f"Test Set MAE: {mae:.4f}")
        print(f"Test Set R2 Score: {r2:.4f}")
        print("-----------------------------")


    except Exception as e:
        print(f"Error during ANFIS prediction or evaluation: {e}")
        df_processed['predicted_energy'] = np.nan # Assign NaN if prediction fails

    return df_processed


# =============================================================================
# Analysis and Saving Functions (Mostly Unchanged, Uses 'predicted_energy')
# =============================================================================

def perform_analysis(df_analyzed):
    """Performs correlation, group analysis, and plotting on daily data."""
    print("\n--- Performing Analysis ---")
    # Ensure prediction and actual columns exist and have valid data
    pred_col = 'predicted_energy'
    actual_col = TARGET_FEATURE # Use the normalized actual energy

    if pred_col in df_analyzed.columns and actual_col in df_analyzed.columns and \
       df_analyzed[pred_col].notna().any() and df_analyzed[actual_col].notna().any():

        # --- Correlation ---
        try:
            correlation = df_analyzed[pred_col].corr(df_analyzed[actual_col])
            print(f"\nCorrelation between {pred_col} and {actual_col}: {correlation:.4f}")
        except Exception as e:
            print(f"Could not calculate correlation: {e}")

        # --- Scatter Plot ---
        print(f"\nGenerating scatter plot (Predicted vs Actual)...\nEnsure '{actual_col}' and '{pred_col}' columns exist.")
        try:
            # Check if columns exist before plotting
            if actual_col not in df_analyzed.columns: print(f"Error: Column '{actual_col}' not found for scatter plot."); raise KeyError
            if pred_col not in df_analyzed.columns: print(f"Error: Column '{pred_col}' not found for scatter plot."); raise KeyError

            plot_sample_df = df_analyzed
            if len(df_analyzed) > SCATTER_PLOT_SAMPLE_SIZE:
                print(f"(Sampling {SCATTER_PLOT_SAMPLE_SIZE} points for plot)")
                plot_sample_df = df_analyzed.sample(n=SCATTER_PLOT_SAMPLE_SIZE, random_state=42)

            plt.figure(figsize=(10, 6))
            sns.scatterplot(data=plot_sample_df, x=actual_col, y=pred_col, alpha=0.6)
            # Add a diagonal line for reference (perfect prediction)
            # Ensure values are valid before finding min/max
            valid_actual = plot_sample_df[actual_col].dropna()
            valid_pred = plot_sample_df[pred_col].dropna()
            if not valid_actual.empty and not valid_pred.empty:
                min_val = min(valid_actual.min(), valid_pred.min())
                max_val = max(valid_actual.max(), valid_pred.max())
                # Handle potential NaN/Inf in min/max_val
                if not (pd.isna(min_val) or pd.isna(max_val) or np.isinf(min_val) or np.isinf(max_val)):
                    plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', label='Perfect Prediction')
                else:
                    print("Warning: Could not determine valid range for diagonal line (NaN/Inf found).")
            else:
                 print("Warning: Could not determine valid range for diagonal line (Empty data after dropna).")


            plt.title('Predicted Energy vs. Normalized Actual Avg Energy (Daily)')
            plt.xlabel('Normalized Actual Avg Energy Per Household')
            plt.ylabel('Predicted Energy (ANFIS Output)')
            plt.grid(True); plt.legend()
            plt.savefig("scatter_plot_daily_pred_vs_actual_xanfis.png"); plt.close()
            print("Scatter plot saved as scatter_plot_daily_pred_vs_actual_xanfis.png.")
        except ImportError: print("Error: Matplotlib or Seaborn not installed.")
        except KeyError: pass # Error message printed above
        except Exception as e: print(f"Error generating scatter plot: {e}")

        # --- Group Analysis by Month ---
        print("\nGroup Analysis by Month:")
        try:
            if 'month' in df_analyzed.columns:
                # Ensure columns exist before grouping
                if actual_col not in df_analyzed.columns: print(f"Error: Column '{actual_col}' not found for monthly analysis."); raise KeyError
                if pred_col not in df_analyzed.columns: print(f"Error: Column '{pred_col}' not found for monthly analysis."); raise KeyError

                monthly_analysis = df_analyzed.groupby('month')[[actual_col, pred_col]].mean()
                print(monthly_analysis)
            else: print("Month column not found for analysis.")
        except KeyError: pass # Error message printed above
        except Exception as e: print(f"Error performing group analysis by month: {e}")

        # --- Time Series Line Plot ---
        print(f"\nGenerating line plot for daily average energy (first {LINE_PLOT_DAYS} days)...")
        try:
            plot_df = df_analyzed.sort_values('day').head(LINE_PLOT_DAYS)
            if not plot_df.empty:
                 # Ensure columns exist before plotting
                if actual_col not in plot_df.columns: print(f"Error: Column '{actual_col}' not found for line plot."); raise KeyError
                if pred_col not in plot_df.columns: print(f"Error: Column '{pred_col}' not found for line plot."); raise KeyError

                plt.figure(figsize=(15, 7))
                plt.plot(plot_df['day'], plot_df[actual_col], label='Actual Avg Energy (Normalized)', marker='.', linestyle='-', alpha=0.7)
                plt.plot(plot_df['day'], plot_df[pred_col], label='Predicted Energy (ANFIS)', marker='x', linestyle='--', alpha=0.7)
                plt.title(f'Daily Avg Energy vs. Prediction (First {LINE_PLOT_DAYS} Days)')
                plt.xlabel('Date'); plt.ylabel('Normalized Value / Prediction')
                plt.legend(); plt.grid(True); plt.xticks(rotation=45); plt.tight_layout()
                plt.savefig("line_plot_daily_time_series_xanfis.png"); plt.close()
                print("Line plot saved as line_plot_daily_time_series_xanfis.png.")
            else: print(f"No data found for the first {LINE_PLOT_DAYS} days.")
        except ImportError: print("Error: Matplotlib not installed.")
        except KeyError: pass # Error message printed above
        except Exception as e: print(f"Error generating line plot: {e}")
    else:
        print(f"\nSkipping analysis: Required columns ('{pred_col}', '{actual_col}') missing or contain only NaNs.")

    print("--- Analysis Complete ---")
    return df_analyzed


def inspect_and_save(df_final):
    """Inspects final results and saves the DataFrame to CSV."""
    global OUTPUT_CSV_PATH
    print("\nInspecting final results sample (Daily Level)...")

    # Define columns to show in the sample output
    columns_to_show_sample = [
        'day', 'avg_energy_per_household', # Original average energy
        TARGET_FEATURE, # Normalized actual energy
        'predicted_energy' # ANFIS prediction
    ]
    # Add some key input features for context
    columns_to_show_sample.extend([col for col in ['temperatureMax', 'humidity', 'month', 'is_holiday', 'previous_day_energy_normalized'] if col in df_final.columns])

    if not df_final.empty:
        print("Sample Daily Data with Predictions:")
        # Ensure columns exist before trying to display them
        valid_cols_to_show = [col for col in columns_to_show_sample if col in df_final.columns]
        print(df_final[valid_cols_to_show].head())

        print("\nPrediction statistics:")
        pred_col = 'predicted_energy'
        actual_norm_col = TARGET_FEATURE

        if pred_col in df_final.columns and pd.api.types.is_numeric_dtype(df_final[pred_col]):
            if df_final[pred_col].notna().any():
                print(f"{pred_col} (ANFIS Output):\n", df_final[pred_col].describe())
            else: print(f"{pred_col} column contains only NaN values.")
        else: print(f"Cannot describe {pred_col} column.")

        if actual_norm_col in df_final.columns and pd.api.types.is_numeric_dtype(df_final[actual_norm_col]):
             if df_final[actual_norm_col].notna().any():
                 print(f"\n{actual_norm_col} (Normalized Actual):\n", df_final[actual_norm_col].describe())
             else: print(f"{actual_norm_col} column contains only NaN values.")
        else: print(f"\nCannot describe {actual_norm_col} column.")

    else: print("DataFrame is empty, cannot show results.")

    # --- Saving ---
    if not df_final.empty:
        try:
            final_columns = df_final.columns.tolist()
            print(f"\nSaving final DataFrame with {len(final_columns)} columns to '{OUTPUT_CSV_PATH}'")
            # print(f"Columns being saved: {final_columns}") # Uncomment to list all columns
            df_final.to_csv(OUTPUT_CSV_PATH, index=False, float_format='%.6f')
            print(f"\nDaily ANFIS predictions generated and saved.")
        except Exception as e: print(f"\nError saving results to CSV: {e}")
    else: print("\nSkipping saving results as the DataFrame is empty.")


# =============================================================================
# Main Execution Flow
# =============================================================================
if __name__ == "__main__":
    # 1. Load Data (including bank holidays from CSV)
    # *** Updated call to load_data ***
    df_daily_dataset, df_bank_holidays, df_info_households, df_weather = load_data()

    # 2. Handle Missing Values
    df_daily_dataset, df_weather = handle_missing_values(df_daily_dataset, df_weather)

    # 3. Process Dates and Holidays (using CSV data)
    # *** Updated call to process_dates_holidays ***
    df_daily_dataset, df_weather, df_bank_holidays = process_dates_holidays(df_daily_dataset, df_weather, df_bank_holidays)

    # 4. Aggregate Energy and Merge Datasets (Daily Level)
    df_merged = aggregate_and_merge(df_daily_dataset, df_weather, df_bank_holidays)

    # 5. Normalize Features (Inputs and Target + Create Lagged)
    df_normalized = normalize_features(df_merged)

    # 6. Visualize Data Distributions (Optional but useful)
    # Ensure features exist before plotting
    features_for_dist_plot = [f for f in ANFIS_INPUT_FEATURES if f in df_normalized.columns]
    if TARGET_FEATURE in df_normalized.columns:
        features_for_dist_plot.append(TARGET_FEATURE)
    visualize_data_distributions(df_normalized, features_for_dist_plot)


    # --- X-ANFIS Section ---
    # 7. Train X-ANFIS model and get predictions
    df_with_predictions = train_predict_xanfis(df_normalized)
    # --- End X-ANFIS Section ---

    # 8. Perform Analysis (using predictions from X-ANFIS)
    df_analyzed = perform_analysis(df_with_predictions)

    # 9. Inspect and Save Results
    inspect_and_save(df_analyzed)

    print("\nScript finished.")


Loading datasets...
Looking for daily energy files matching: .\daily_dataset\block_*.csv
Found 112 daily energy block files. Loading and concatenating...


Loading block files:   0%|          | 0/112 [00:00<?, ?it/s]

Combined daily dataset loaded successfully (3510433 rows).
Core datasets loaded successfully.

Handling missing values...
Dropped 30 rows with missing energy data from original daily data.
Dropped 1 rows with missing weather data based on columns: ['temperatureMax', 'temperatureMin', 'dewPoint', 'humidity', 'cloudCover', 'windSpeed', 'pressure'].

Processing dates and holidays...
Processing bank holidays from CSV...
Processed 25 unique bank holiday dates from CSV.
Date conversions complete.

Aggregating daily energy and merging datasets...
Calculating daily average energy per household...


C:\Users\mrkun\AppData\Local\Temp\ipykernel_6720\1081376306.py:196: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_daily_agg['avg_energy_per_household'].replace([np.inf, -np.inf], np.nan, inplace=True)
C:\Users\mrkun\AppData\Local\Temp\ipykernel_6720\1081376306.py:238: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are settin

Daily average energy calculated for 829 days.
Merging daily aggregated energy with weather and holidays...
Rows after merging aggregated energy with weather: 826
Rows after adding holidays: 826
Added 'month' column.

Total rows in final merged DataFrame (Daily Level): 826
Columns: ['day', 'total_energy_sum', 'household_count', 'avg_energy_per_household', 'temperatureMax', 'temperatureMin', 'dewPoint', 'humidity', 'cloudCover', 'windSpeed', 'pressure', 'is_holiday', 'month']

Normalizing features...
Normalizing target feature 'actual_energy'...
Target feature 'avg_energy_per_household' normalized to 'actual_energy_normalized'.
Created lagged feature 'previous_day_energy_normalized'.
Input features to normalize (0-1 scale): ['temperatureMax', 'temperatureMin', 'dewPoint', 'humidity', 'cloudCover', 'windSpeed', 'pressure', 'month', 'is_holiday', 'previous_day_energy_normalized']
All input features normalization complete.

No NaNs found in final ANFIS input features or target column.

Gene